# ScrapLink — Sprint 2: Price Recommendation Model (Research Edition)
**AI-Driven B2B Industrial Scrap Exchange Platform**

## Research Objective
Compare **3 training strategies** for XGBoost on commodity price time series:

| Approach | Target Variable | Core Idea |
|---|---|---|
| **A1 — Raw Price** | `spot_price_inr_kg` | Predict absolute price *(non-stationary baseline)* |
| **A2 — First Difference** | `p_t − p_{t-1}` | Predict daily price change *(stationary)* |
| **A3 — Log Return** ⭐ | `log(p_t / p_{t-1})` | Predict % change in log space *(stationary + scale-invariant)* |

> **Why this matters:** Tree-based ensemble models cannot extrapolate beyond their training range.
> Training on raw non-stationary prices causes a prediction ceiling — visible in our residual charts.
> Approaches A2 and A3 fix this by making the target stationary.

## 1. Install Dependencies

In [ ]:
import subprocess, sys
for pkg in ['xgboost','scikit-learn','pandas','numpy','matplotlib','seaborn','joblib']:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',pkg])
print('All packages ready!')

## 2. Imports & Configuration

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

# ── CONFIG ─────────────────────────────────────────────────────────────
# Update FEATURE_CSV to your path (upload engineered_commodity_features.csv)
FEATURE_CSV  = 'engineered_commodity_features.csv'  # <── CHANGE IF NEEDED
MODELS_DIR   = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

FEATURE_COLS = [
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'rolling_mean_7', 'rolling_mean_30',
    'rolling_std_7',  'rolling_std_30',
    'pct_change_7',   'pct_change_30',
]
TARGET_COL = 'spot_price_inr_kg'
SEED = 42
print('Config OK')

## 3. Load Data

In [ ]:
df = pd.read_csv(FEATURE_CSV)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['commodity','date']).reset_index(drop=True)

print(f'Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range  : {df.date.min().date()} to {df.date.max().date()}')
print(f'Commodities : {sorted(df.commodity.unique())}')
df.head(3)

## 4. EDA — Understanding the Data

In [ ]:
stats = (
    df.groupby('commodity')[TARGET_COL]
    .agg(['mean','std','min','max'])
    .round(2)
)
stats.columns = ['Mean (Rs/kg)','Std Dev','Min','Max']
stats['Volatility%'] = (stats['Std Dev']/stats['Mean (Rs/kg)']*100).round(1)
stats.sort_values('Volatility%', ascending=False)

In [ ]:
commodities = sorted(df.commodity.unique())
cols = 3; rows = (len(commodities)+cols-1)//cols
fig, axes = plt.subplots(rows, cols, figsize=(18, rows*4))
axes = axes.flatten()
for i, comm in enumerate(commodities):
    sub = df[df.commodity==comm]
    axes[i].plot(sub['date'], sub[TARGET_COL], lw=1.2, color='steelblue')
    axes[i].fill_between(sub['date'], sub[TARGET_COL], alpha=0.12, color='steelblue')
    axes[i].set_title(comm, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Rs/kg')
    axes[i].tick_params(axis='x', rotation=30)
for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig.suptitle('Commodity Spot Price Time Series — Prices Trend UPWARD (Non-Stationary!)',
             fontsize=13, fontweight='bold', color='crimson', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'price_timeseries.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Compare raw price vs log return for one commodity — shows stationarity
copper = df[df.commodity=='Copper'].sort_values('date').copy()
copper['log_return'] = np.log(copper[TARGET_COL] / copper[TARGET_COL].shift(1))
copper['first_diff'] = copper[TARGET_COL].diff()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(copper['date'], copper[TARGET_COL], color='steelblue', lw=1)
axes[0].set_title('A1: Raw Price — NON-STATIONARY\n(trends upward, XGBoost ceiling problem)', fontweight='bold', color='crimson')
axes[0].set_ylabel('Rs/kg')

axes[1].plot(copper['date'], copper['first_diff'], color='darkorange', lw=0.8, alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_title('A2: First Difference — STATIONARY\n(oscillates around 0)', fontweight='bold', color='green')
axes[1].set_ylabel('Δp (Rs/kg)')

axes[2].plot(copper['date'], copper['log_return'], color='purple', lw=0.8, alpha=0.8)
axes[2].axhline(0, color='black', lw=0.8, ls='--')
axes[2].set_title('A3: Log Return — STATIONARY + Scale-Invariant\n(bounded range, best for XGBoost)', fontweight='bold', color='green')
axes[2].set_ylabel('log(p_t / p_{t-1})')

fig.suptitle('Copper — Why Target Transformation Matters for Tree Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'stationarity_comparison.png'), bbox_inches='tight')
plt.show()

In [ ]:
vol = stats['Volatility%'].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12,5))
bars = ax.barh(vol.index, vol.values, color=sns.color_palette('muted', len(vol)))
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('Volatility (Std/Mean × 100)')
ax.set_title('Volatility by Commodity — Copper (40.8%) vs PET (7.5%) = 5x difference\nJustifies per-commodity training', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'volatility_chart.png'), bbox_inches='tight')
plt.show()

In [ ]:
pivot = df.pivot_table(index='date', columns='commodity', values=TARGET_COL, aggfunc='mean')
corr = pivot.corr().round(2)
fig, ax = plt.subplots(figsize=(12,9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink':0.7})
ax.set_title('Commodity Price Correlation Matrix\nNote: Copper-Brass (-0.74) move opposite → mixing them in 1 model is wrong',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'correlation_heatmap.png'), bbox_inches='tight')
plt.show()

## 5. Helper Functions & Shared Split Logic
All 3 approaches share the **same 80/20 time-aware split** (no shuffle) for fair comparison.

In [ ]:
def compute_metrics(y_true, y_pred, name):
    """Returns MAE, RMSE, R2 dict."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = 1 - np.sum((y_true-y_pred)**2) / np.sum((y_true-y_true.mean())**2)
    print(f'  {name:<40}  MAE={mae:7.2f}  RMSE={rmse:8.2f}  R2={r2:.4f}')
    return {'approach': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R2': round(r2,4)}

def plot_actual_vs_pred(y_true, y_pred, title, ax, color):
    lims = [min(y_true.min(), y_pred.min())-10, max(y_true.max(), y_pred.max())+10]
    ax.scatter(y_true, y_pred, alpha=0.2, s=8, color=color)
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual Price (Rs/kg)'); ax.set_ylabel('Predicted (Rs/kg)')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)

def plot_residuals(y_true, y_pred, title, ax, color):
    residuals = y_true - y_pred
    ax.scatter(y_pred, residuals, alpha=0.2, s=8, color=color)
    ax.axhline(0, color='red', lw=1.5, ls='--')
    ax.set_xlabel('Predicted Price (Rs/kg)'); ax.set_ylabel('Residual')
    ax.set_title(title, fontweight='bold')

# ── Prepare clean dataset (drop NaN from lag_30) ──────────────────────────
df_clean = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).sort_values(['commodity','date']).reset_index(drop=True)
print(f'Clean rows: {len(df_clean):,}')

# All results will be collected here
ALL_RESULTS = []

---
## Approach 1 — Raw Price (Non-Stationary Baseline)
> **Target:** `spot_price_inr_kg` — absolute price

> **Expected problem:** XGBoost cannot predict prices beyond its training range → prediction ceiling in test set.

> We train 3 sub-variants to isolate each improvement:
> - A1a: Linear Regression (global)
> - A1b: XGBoost (global, all commodities mixed)
> - A1c: XGBoost (per-commodity)

In [ ]:
# Global split (80/20 time-aware)
df_a1 = df_clean.copy()
split = int(len(df_a1)*0.8)
train_a1, test_a1 = df_a1.iloc[:split], df_a1.iloc[split:]

X_train_a1 = train_a1[FEATURE_COLS].values
y_train_a1 = train_a1[TARGET_COL].values
X_test_a1  = test_a1[FEATURE_COLS].values
y_test_a1  = test_a1[TARGET_COL].values

scaler_a1 = StandardScaler()
X_train_sc = scaler_a1.fit_transform(X_train_a1)
X_test_sc  = scaler_a1.transform(X_test_a1)

print(f'Train: {len(X_train_a1):,} | Test: {len(X_test_a1):,}')
print(f'Train price range: Rs {y_train_a1.min():.0f} – Rs {y_train_a1.max():.0f}')
print(f'Test  price range: Rs {y_test_a1.min():.0f} – Rs {y_test_a1.max():.0f}')

In [ ]:
# A1a — Linear Regression (global)
lr = LinearRegression().fit(X_train_sc, y_train_a1)
y_pred_lr = lr.predict(X_test_sc)
r_a1a = compute_metrics(y_test_a1, y_pred_lr, 'A1a: Linear Regression (global)')
ALL_RESULTS.append(r_a1a)

In [ ]:
# A1b — XGBoost (global, all commodities mixed)
xgb_a1b = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=SEED,
                        n_jobs=-1, verbosity=0)
xgb_a1b.fit(X_train_a1, y_train_a1, eval_set=[(X_test_a1, y_test_a1)], verbose=False)
y_pred_a1b = xgb_a1b.predict(X_test_a1)
r_a1b = compute_metrics(y_test_a1, y_pred_a1b, 'A1b: XGBoost global (raw price)')
ALL_RESULTS.append(r_a1b)

In [ ]:
# A1c — XGBoost PER COMMODITY (raw price)
models_a1c = {}
preds_a1c  = np.zeros(len(test_a1))

for comm in df_a1['commodity'].unique():
    sub       = df_a1[df_a1['commodity']==comm].sort_values('date')
    n         = len(sub)
    sp        = int(n*0.8)
    X_tr, X_te = sub.iloc[:sp][FEATURE_COLS].values, sub.iloc[sp:][FEATURE_COLS].values
    y_tr, y_te = sub.iloc[:sp][TARGET_COL].values,   sub.iloc[sp:][TARGET_COL].values
    if len(X_tr) < 10: continue
    m = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, random_state=SEED,
                     n_jobs=-1, verbosity=0)
    m.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
    models_a1c[comm] = m

# Reconstruct global test predictions from per-commodity models
test_preds_a1c = []
test_true_a1c  = []
for comm, model in models_a1c.items():
    sub  = df_a1[df_a1['commodity']==comm].sort_values('date')
    sp   = int(len(sub)*0.8)
    X_te = sub.iloc[sp:][FEATURE_COLS].values
    y_te = sub.iloc[sp:][TARGET_COL].values
    test_preds_a1c.extend(model.predict(X_te).tolist())
    test_true_a1c.extend(y_te.tolist())

y_true_a1c = np.array(test_true_a1c)
y_pred_a1c = np.array(test_preds_a1c)
r_a1c = compute_metrics(y_true_a1c, y_pred_a1c, 'A1c: XGBoost per-commodity (raw price)')
ALL_RESULTS.append(r_a1c)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_actual_vs_pred(y_test_a1,  y_pred_lr,    'A1a LR — Actual vs Predicted',            axes[0,0], '#6baed6')
plot_actual_vs_pred(y_test_a1,  y_pred_a1b,   'A1b XGBoost Global — Actual vs Predicted', axes[0,1], '#2171b5')
plot_actual_vs_pred(y_true_a1c, y_pred_a1c,   'A1c XGBoost Per-Commodity — Actual vs Predicted', axes[0,2], '#08519c')
plot_residuals(y_test_a1,  y_pred_lr,    'A1a LR — Residuals',            axes[1,0], '#6baed6')
plot_residuals(y_test_a1,  y_pred_a1b,   'A1b XGBoost Global — Residuals', axes[1,1], '#2171b5')
plot_residuals(y_true_a1c, y_pred_a1c,   'A1c XGBoost Per-Comm — Residuals',  axes[1,2], '#08519c')
fig.suptitle('Approach 1 — Raw Price (Non-Stationary) | Note XGBoost ceiling in A1b',
             fontsize=13, fontweight='bold', color='crimson')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'a1_raw_price_plots.png'), bbox_inches='tight')
plt.show()

---
## Approach 2 — First Difference (Stationary Target)
> **Target:** `Δp = p_t − p_{t-1}` — daily price change in Rs/kg

> **Reconstruction:** `predicted_price = lag_1 + predicted_Δp`

> Since Δp is always a small bounded number (~±50 Rs), XGBoost can predict any future price level.

In [ ]:
models_a2 = {}
test_preds_a2 = []
test_true_a2  = []

for comm in df_clean['commodity'].unique():
    sub = df_clean[df_clean['commodity']==comm].sort_values('date').copy()
    # Target = first difference
    sub['target_diff'] = sub[TARGET_COL].diff()
    sub = sub.dropna(subset=['target_diff']).reset_index(drop=True)
    if len(sub) < 20: continue

    sp   = int(len(sub)*0.8)
    X_tr = sub.iloc[:sp][FEATURE_COLS].values
    y_tr = sub.iloc[:sp]['target_diff'].values
    X_te = sub.iloc[sp:][FEATURE_COLS].values
    y_te_diff  = sub.iloc[sp:]['target_diff'].values
    lag1_te    = sub.iloc[sp:]['lag_1'].values        # needed for reconstruction
    y_te_price = sub.iloc[sp:][TARGET_COL].values    # ground truth prices

    m = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     min_child_weight=3, random_state=SEED,
                     n_jobs=-1, verbosity=0)
    m.fit(X_tr, y_tr, eval_set=[(X_te, y_te_diff)], verbose=False)
    models_a2[comm] = m

    # Reconstruct price: lag_1 + predicted_delta
    pred_diff  = m.predict(X_te)
    pred_price = lag1_te + pred_diff

    test_preds_a2.extend(pred_price.tolist())
    test_true_a2.extend(y_te_price.tolist())

y_true_a2 = np.array(test_true_a2)
y_pred_a2 = np.array(test_preds_a2)
r_a2 = compute_metrics(y_true_a2, y_pred_a2, 'A2: XGBoost per-comm (first difference)')
ALL_RESULTS.append(r_a2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_actual_vs_pred(y_true_a2, y_pred_a2, 'A2 First Diff — Actual vs Predicted\n(No ceiling!)', axes[0], 'darkorange')
plot_residuals(y_true_a2, y_pred_a2, 'A2 First Diff — Residuals', axes[1], 'darkorange')
fig.suptitle('Approach 2 — First Difference Target (Stationary)', fontsize=13, fontweight='bold', color='green')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'a2_first_diff_plots.png'), bbox_inches='tight')
plt.show()

---
## Approach 3 — Log Return ⭐ (Best Practice)
> **Target:** `r_t = log(p_t / p_{t-1})` — log percentage change

> **Reconstruction:** `predicted_price = lag_1 × exp(predicted_r_t)`

> **Why log return is better than first difference:**
> - Scale-invariant: a ₹10 move in Paper (₹18/kg) ≠ a ₹10 move in Copper (₹650/kg)
> - Log returns are symmetric: +10% then -10% = same magnitude
> - Standard approach in quantitative finance for price modelling

In [ ]:
models_a3 = {}
test_preds_a3 = []
test_true_a3  = []

for comm in df_clean['commodity'].unique():
    sub = df_clean[df_clean['commodity']==comm].sort_values('date').copy()
    # Target = log return
    sub['log_return'] = np.log(sub[TARGET_COL] / sub[TARGET_COL].shift(1))
    sub = sub.dropna(subset=['log_return']).reset_index(drop=True)
    if len(sub) < 20: continue

    sp   = int(len(sub)*0.8)
    X_tr = sub.iloc[:sp][FEATURE_COLS].values
    y_tr = sub.iloc[:sp]['log_return'].values
    X_te = sub.iloc[sp:][FEATURE_COLS].values
    y_te_logret   = sub.iloc[sp:]['log_return'].values
    lag1_te       = sub.iloc[sp:]['lag_1'].values
    y_te_price    = sub.iloc[sp:][TARGET_COL].values

    m = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.03,
                     subsample=0.8, colsample_bytree=0.8,
                     min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
                     random_state=SEED, n_jobs=-1, verbosity=0)
    m.fit(X_tr, y_tr, eval_set=[(X_te, y_te_logret)], verbose=False)
    models_a3[comm] = m

    # Reconstruct price: lag_1 * exp(predicted_log_return)
    pred_logret = m.predict(X_te)
    pred_price  = lag1_te * np.exp(pred_logret)

    test_preds_a3.extend(pred_price.tolist())
    test_true_a3.extend(y_te_price.tolist())

y_true_a3 = np.array(test_true_a3)
y_pred_a3 = np.array(test_preds_a3)
r_a3 = compute_metrics(y_true_a3, y_pred_a3, 'A3: XGBoost per-comm (log return) ⭐')
ALL_RESULTS.append(r_a3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_actual_vs_pred(y_true_a3, y_pred_a3, 'A3 Log Return — Actual vs Predicted\n(No ceiling, scale-invariant!)', axes[0], 'purple')
plot_residuals(y_true_a3, y_pred_a3, 'A3 Log Return — Residuals', axes[1], 'purple')
fig.suptitle('Approach 3 — Log Return Target (Stationary + Scale-Invariant) ⭐', fontsize=13, fontweight='bold', color='green')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'a3_log_return_plots.png'), bbox_inches='tight')
plt.show()

---
## 6. Research Comparison — All Approaches

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS).set_index('approach')
best_mae  = results_df['MAE'].idxmin()
best_rmse = results_df['RMSE'].idxmin()
print('='*75)
print('  SPRINT 2 — RESEARCH COMPARISON: ALL APPROACHES')
print('='*75)
print(f"  {'Approach':<45} {'MAE':>10} {'RMSE':>10} {'R2':>8}")
print(f"  {'-'*45} {'-'*10} {'-'*10} {'-'*8}")
for name, row in results_df.iterrows():
    star = ' ⭐ BEST' if name == best_rmse else ''
    print(f"  {name:<45} Rs{row['MAE']:>7.2f} Rs{row['RMSE']:>7.2f} {row['R2']:>8.4f}{star}")
print('='*75)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#6baed6','#2171b5','#08519c','darkorange','purple']
short_names = [
    'A1a\nLR Global', 'A1b\nXGB Global', 'A1c\nXGB/Comm',
    'A2\nFirst Diff', 'A3\nLog Return'
]

for ax, metric in zip(axes, ['MAE','RMSE']):
    vals = results_df[metric].values
    bars = ax.bar(short_names, vals, color=colors, edgecolor='white', width=0.6)
    ax.bar_label(bars, labels=[f'Rs {v:.1f}' for v in vals], padding=4, fontsize=9)
    ax.set_title(f'{metric} (Rs/kg) — Lower is Better', fontweight='bold')
    ax.set_ylabel(f'{metric} (Rs/kg)')
    ax.set_ylim(0, max(vals)*1.25)
    # Highlight best
    best_idx = list(vals).index(min(vals))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

fig.suptitle('Research Comparison: 5 Approaches — All 3 Training Strategies\n(Gold border = best per metric)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'research_comparison_bar.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side actual vs predicted: best of each approach
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_actual_vs_pred(y_true_a1c, y_pred_a1c,
    f'A1c: XGB per-comm Raw Price\nMAE=Rs{r_a1c["MAE"]} RMSE=Rs{r_a1c["RMSE"]}',
    axes[0], '#08519c')

plot_actual_vs_pred(y_true_a2, y_pred_a2,
    f'A2: XGB First Diff\nMAE=Rs{r_a2["MAE"]} RMSE=Rs{r_a2["RMSE"]}',
    axes[1], 'darkorange')

plot_actual_vs_pred(y_true_a3, y_pred_a3,
    f'A3: XGB Log Return ⭐\nMAE=Rs{r_a3["MAE"]} RMSE=Rs{r_a3["RMSE"]}',
    axes[2], 'purple')

fig.suptitle('Best of Each Approach — Actual vs Predicted\nNote: A2 and A3 have NO ceiling because target is stationary',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'compare_actual_vs_predicted.png'), bbox_inches='tight')
plt.show()

## 7. Feature Importance — Best Model (A3: Log Return)

In [ ]:
# Average feature importance across all per-commodity models of A3
fi_all = np.zeros(len(FEATURE_COLS))
for m in models_a3.values():
    fi_all += m.feature_importances_
fi_all /= len(models_a3)
fi_series = pd.Series(fi_all, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10,5))
fi_series.plot(kind='barh', ax=ax, color=sns.color_palette('Blues_r', len(fi_series)))
for i, v in enumerate(fi_series):
    ax.text(v+0.002, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('Mean Feature Importance (across all commodities)')
ax.set_title('A3 Log Return — Feature Importance (Averaged Per-Commodity Models)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR,'feature_importance_a3.png'), bbox_inches='tight')
plt.show()

## 8. Save Best Model Artifacts

In [ ]:
# Determine best approach by RMSE
best_approach = results_df['RMSE'].idxmin()
print(f'Best approach: {best_approach}')

# Save all A3 per-commodity models (expected best)
joblib.dump(models_a3, os.path.join(MODELS_DIR, 'xgboost_per_commodity_logreturn.pkl'))
joblib.dump(models_a1c, os.path.join(MODELS_DIR, 'xgboost_per_commodity_rawprice.pkl'))
joblib.dump(models_a2, os.path.join(MODELS_DIR, 'xgboost_per_commodity_firstdiff.pkl'))

report = {
    'sprint': 'Sprint 2 — Research Comparison',
    'feature_cols': FEATURE_COLS,
    'approaches': ALL_RESULTS,
    'best_by_rmse': best_approach,
    'note': 'A3 log-return approach is production-recommended for non-stationary price series',
}
with open(os.path.join(MODELS_DIR,'research_evaluation_report.json'),'w') as f:
    json.dump(report, f, indent=2)

print('Artifacts saved:')
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(os.path.join(MODELS_DIR,f))
    print(f'  {f:<50} {size:>8,} bytes')

## 9. Live Inference Demo — Best Model (A3 Log Return)

In [ ]:
def predict_price_logreturn(commodity, lag_1, lag_7, lag_14, lag_30,
                             rolling_mean_7, rolling_mean_30,
                             rolling_std_7, rolling_std_30,
                             pct_change_7, pct_change_30,
                             confidence_rmse=None):
    """
    A3 inference: predicts log return then reconstructs price.
    lag_1 is critical — it is the 'previous price' used for reconstruction.
    """
    loaded = joblib.load(os.path.join(MODELS_DIR,'xgboost_per_commodity_logreturn.pkl'))
    if commodity not in loaded:
        raise ValueError(f'Unknown commodity: {commodity}. Known: {list(loaded.keys())}')
    model = loaded[commodity]

    feats = np.array([[lag_1, lag_7, lag_14, lag_30,
                       rolling_mean_7, rolling_mean_30,
                       rolling_std_7, rolling_std_30,
                       pct_change_7, pct_change_30]])

    log_ret = float(model.predict(feats)[0])
    pred    = round(lag_1 * np.exp(log_ret), 2)

    # Confidence band using commodity-specific volatility from EDA
    comm_std = stats.loc[commodity, 'Std Dev'] if confidence_rmse is None else confidence_rmse
    lower = round(max(pred - 1.96*comm_std, 0), 2)
    upper = round(pred + 1.96*comm_std, 2)

    trend = 'UPWARD' if log_ret > 0.001 else ('DOWNWARD' if log_ret < -0.001 else 'STABLE')

    return {
        'commodity':         commodity,
        'recommended_price': pred,
        'price_band':        {'lower': lower, 'upper': upper},
        'log_return_pred':   round(log_ret, 6),
        'trend':             trend,
        'unit':              'INR/kg',
        'approach':          'A3 — Log Return XGBoost (per-commodity)',
        'disclaimer':        'AI-estimated reference. Not a guaranteed transaction price.',
    }

# Demo for all commodities
rows = []
for comm in sorted(models_a3.keys()):
    latest = df_clean[df_clean['commodity']==comm].sort_values('date').iloc[-1]
    res = predict_price_logreturn(
        commodity=comm,
        lag_1=latest['lag_1'], lag_7=latest['lag_7'],
        lag_14=latest['lag_14'], lag_30=latest['lag_30'],
        rolling_mean_7=latest['rolling_mean_7'],   rolling_mean_30=latest['rolling_mean_30'],
        rolling_std_7=latest['rolling_std_7'],     rolling_std_30=latest['rolling_std_30'],
        pct_change_7=latest['pct_change_7'],       pct_change_30=latest['pct_change_30'],
    )
    rows.append({'Commodity':res['commodity'],
                 'Recommended (Rs/kg)':res['recommended_price'],
                 'Lower Band':res['price_band']['lower'],
                 'Upper Band':res['price_band']['upper'],
                 'Trend':res['trend'],
                 'Log Return':res['log_return_pred']})

out = pd.DataFrame(rows).set_index('Commodity')
print('=== Live Recommendations — A3 Log Return Model ===')
out